In [2]:
# =========================================================
# Isolation Forest - 이상 탐지 실험
# 데이터셋: NSL-KDD, UNSW-NB15
# 전처리  : MinMax / Quantile
# 임계값  : Bootstrap (α=0.10, α=0.15)
# 범주형  : Binary Encoding (category_encoders)
# =========================================================

import numpy as np
import pandas as pd
from category_encoders import BinaryEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, confusion_matrix
)

# =========================================================
# 경로 설정
# =========================================================
DATA_DIR = "./data/"

NSL_CAT  = ["protocol_type", "service", "flag"]
UNSW_CAT = ["proto", "service", "state"]

# =========================================================
# Binary Encoding
# =========================================================
def binary_encode(df, cat_cols, encoder=None, fit=True):
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df, encoder
    if encoder is None:
        encoder = BinaryEncoder(cols=existing, return_df=True)
    encoded_df = encoder.fit_transform(df) if fit else encoder.transform(df)
    return encoded_df, encoder

def load_and_encode(csv_path, cat_cols, label_col, encoder=None, fit=True):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    label_series = df.pop(label_col) if label_col in df.columns else None
    df_encoded, encoder = binary_encode(df, cat_cols, encoder=encoder, fit=fit)
    if label_series is not None:
        df_encoded[label_col] = label_series.values
    return df_encoded, encoder

def encode_label(df, label_col="class"):
    if label_col in df.columns and df[label_col].dtype == object:
        df = df.copy()
        df[label_col] = (df[label_col].str.strip().str.lower() != "normal").astype(int)
    return df

# =========================================================
# Bootstrap 임계값
# =========================================================
def bootstrap_threshold(scores, percentiles=range(0, 101), B=500, seed=42):
    scores = np.asarray(scores).ravel()
    rng    = np.random.default_rng(seed)
    n      = len(scores)
    boot   = {p: np.empty(B, dtype=np.float32) for p in percentiles}
    for b in range(B):
        sample = rng.choice(scores, size=n, replace=True)
        for p in percentiles:
            boot[p][b] = np.percentile(sample, p)
    df = pd.DataFrame(
        {p: float(np.median(boot[p])) for p in percentiles}.items(),
        columns=["Percentile", "Threshold"]
    ).set_index("Percentile")
    return df

# =========================================================
# 성능 평가
# =========================================================
def evaluate(test_scores, y_true, thresholds_dict):
    try:
        auc = roc_auc_score(y_true, test_scores)
    except ValueError:
        auc = np.nan

    rows = []
    for alpha, T in thresholds_dict.items():
        y_pred = (test_scores >= T).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        rows.append({
            "α":           alpha,
            "Precision":   precision_score(y_true, y_pred, zero_division=0),
            "Recall":      recall_score(y_true, y_pred, zero_division=0),
            "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
            "F1-score":    f1_score(y_true, y_pred, zero_division=0),
            "Accuracy":    accuracy_score(y_true, y_pred),
            "AUC":         auc,
        })
    return pd.DataFrame(rows).sort_values("α").reset_index(drop=True)

# =========================================================
# 파이프라인
# =========================================================
def run_pipeline(train_path, valid_path, test_path,
                 cat_cols, label_col, seed=42):

    # 로드 & 인코딩 (train 기준 fit)
    train_df, encoder = load_and_encode(train_path, cat_cols, label_col, fit=True)
    valid_df, _       = load_and_encode(valid_path, cat_cols, label_col, encoder=encoder, fit=False)
    test_df,  _       = load_and_encode(test_path,  cat_cols, label_col, encoder=encoder, fit=False)

    # NSL-KDD 레이블 인코딩 (문자열 → 0/1)
    if label_col == "class":
        train_df = encode_label(train_df, label_col)
        valid_df = encode_label(valid_df, label_col)
        test_df  = encode_label(test_df,  label_col)

    # 모델 학습
    X_train = train_df.drop(columns=[label_col], errors="ignore")
    model   = IsolationForest(
        n_estimators=100, max_samples=512, max_features=0.4,
        contamination="auto", random_state=seed
    )
    model.fit(X_train)

    # valid 이상 점수로 Bootstrap 임계값 산출
    X_valid      = valid_df.drop(columns=[label_col], errors="ignore")
    valid_scores = -model.score_samples(X_valid)
    df_thr       = bootstrap_threshold(valid_scores, seed=seed)

    # α=0.10 → P90, α=0.15 → P85
    thresholds = {0.10: float(df_thr.loc[90, "Threshold"]),
                  0.15: float(df_thr.loc[85, "Threshold"])}

    # test 평가
    X_test      = test_df.drop(columns=[label_col], errors="ignore")
    test_scores = -model.score_samples(X_test)
    y_true      = test_df[label_col].values.astype(int)

    return evaluate(test_scores, y_true, thresholds)

# =========================================================
# 결과 출력
# =========================================================
def print_results(dataset_name, df_mm, df_qt):
    COLS    = ["Precision", "Recall", "Specificity", "F1-score", "Accuracy", "AUC"]
    COLS_KR = ["정밀도",    "민감도",  "특이도",      "F1 점수",  "정확도",   "AUC"]
    W       = 78

    print("=" * W)
    print(f" {dataset_name}")
    print("=" * W)
    print(f"  {'':22s}" + "".join(f"{k:>8}" for k in COLS_KR))
    print("-" * W)

    for scaler, df in [("MinMax", df_mm), ("Quantile", df_qt)]:
        for _, row in df.iterrows():
            label = f"  {scaler:<10} α={row['α']:.2f}"
            vals  = "".join(f"{row[c]:>8.2f}" for c in COLS)
            print(f"{label:<26}{vals}")
        print("-" * W)
    print()

# =========================================================
# 실험 실행
# =========================================================
if __name__ == "__main__":

    # ── NSL-KDD ──────────────────────────────────────────
    res_nsl_mm = run_pipeline(
        DATA_DIR + "NSL_KDD_MinMax_train_normal_80.csv",
        DATA_DIR + "NSL_KDD_MinMax_train_normal_20.csv",
        DATA_DIR + "NSL_KDD_MinMax_test.csv",
        cat_cols=NSL_CAT, label_col="class"
    )
    res_nsl_qt = run_pipeline(
        DATA_DIR + "NSL_KDD_Quantile_train_normal_80.csv",
        DATA_DIR + "NSL_KDD_Quantile_train_normal_20.csv",
        DATA_DIR + "NSL_KDD_Quantile_test.csv",
        cat_cols=NSL_CAT, label_col="class"
    )

    # ── UNSW-NB15 ────────────────────────────────────────
    res_unsw_mm = run_pipeline(
        DATA_DIR + "UNSW_NB15_MinMax_train_normal_80.csv",
        DATA_DIR + "UNSW_NB15_MinMax_train_normal_20.csv",
        DATA_DIR + "UNSW_NB15_MinMax_test.csv",
        cat_cols=UNSW_CAT, label_col="label"
    )
    res_unsw_qt = run_pipeline(
        DATA_DIR + "UNSW_NB15_Quantile_train_normal_80.csv",
        DATA_DIR + "UNSW_NB15_Quantile_train_normal_20.csv",
        DATA_DIR + "UNSW_NB15_Quantile_test.csv",
        cat_cols=UNSW_CAT, label_col="label"
    )

    # ── 출력 ─────────────────────────────────────────────
    print_results("NSL-KDD",   res_nsl_mm,  res_nsl_qt)
    print_results("UNSW-NB15", res_unsw_mm, res_unsw_qt)

 NSL-KDD
                             정밀도     민감도     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------------
  MinMax     α=0.10           0.92    0.76    0.92    0.83    0.83    0.95
  MinMax     α=0.15           0.91    0.80    0.90    0.85    0.84    0.95
------------------------------------------------------------------------------
  Quantile   α=0.10           0.96    0.82    0.95    0.89    0.88    0.97
  Quantile   α=0.15           0.95    0.88    0.93    0.91    0.90    0.97
------------------------------------------------------------------------------

 UNSW-NB15
                             정밀도     민감도     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------------
  MinMax     α=0.10           0.78    0.51    0.83    0.62    0.65    0.79
  MinMax     α=0.15           0.77    0.69    0.75    0.73    0.72    0.79
-------------------------------------------------------------------